In [2]:
!pip install librosa
import os
import pandas as pd
import librosa
import numpy as np
import matplotlib.pyplot as plt
import librosa.display


  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached audioread-3.1.0-py3-none-any.whl (23 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 1.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 140.4 kB/s  0:03:11m0:00:0200:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 159.6 kB/s  0:00:06 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [librosa]m8/9 [librosa]e]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# 1. Setup Folders - Fixed naming to match the loop logic
output_base = "../data/heart_grades_classified"
# Note: I changed "Grade_3_Plus" to "Grade_3" to match the min(val, 3) logic
grades = ["Grade_0", "Grade_1", "Grade_2", "Grade_3"]
for g in grades:
    os.makedirs(os.path.join(output_base, g), exist_ok=True)


In [4]:
# 2. Metadata Parser - Added more print statements to debug
def get_severity(pid, raw_dir):
    txt_path = os.path.join(raw_dir, f"{pid}.txt")
    if not os.path.exists(txt_path): return 0

    with open(txt_path, 'r') as f:
        for line in f:
            if "#Systolic murmur grading:" in line:
                grade_str = line.split(":")[-1].strip()
                if grade_str == "nan": return 0
                mapping = {'I/VI': 1, 'II/VI': 2, 'III/VI': 3, 'IV/VI': 4, 'V/VI': 5, 'VI/VI': 6}
                val = mapping.get(grade_str, 0)
                return min(val, 3) # Maps 3, 4, 5, 6 all to Grade 3
    return 0

In [ ]:
# 3.  Spectrogram Generator
def save_melspec(y, sr, save_path):
    if len(y) < 2048:
        y = librosa.util.pad_center(y, size=2048)

    
    fig = plt.figure(figsize=(2, 2))
    ax = fig.add_subplot(111)

    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
    S_dB = librosa.power_to_db(S, ref=np.max)

    librosa.display.specshow(S_dB, sr=sr, ax=ax)
    plt.axis('off')

    # Save figure without extra whitespace
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig) 
    plt.close('all') 

In [8]:
# 4. Main Processing Loop
raw_dir = "../exported_models/heart_grades"
patient_ids = set([f.split('.')[0] for f in os.listdir(raw_dir) if f.endswith('.txt')])

print(f"Processing {len(patient_ids)} patients...")

# Limit for testing - remove [:100] to process everything
for pid in list(patient_ids):
    grade = get_severity(pid, raw_dir)
    target_folder = os.path.join(output_base, f"Grade_{grade}")

    for loc in ["MV", "AV"]:
        wav_path = os.path.join(raw_dir, f"{pid}_{loc}.wav")
        tsv_path = os.path.join(raw_dir, f"{pid}_{loc}.tsv")

        if os.path.exists(wav_path) and os.path.exists(tsv_path):
            try:
                y, sr = librosa.load(wav_path, sr=4000)
                tsv = pd.read_csv(tsv_path, sep='\t', header=None)

                counter = 0
                for _, row in tsv.iterrows():
                    # State 2 = Systole
                    if row[2] == 2:
                        start, end = int(row[0]*sr), int(row[1]*sr)
                        segment = y[start:end]

                        if len(segment) > 100:
                            save_name = f"{pid}_{loc}_seg_{counter}.png"
                            save_path = os.path.join(target_folder, save_name)
                            save_melspec(segment, sr, save_path)
                            counter += 1
            except Exception as e:
                continue

print("\n--- Feature Extraction Complete ---")
for g in grades:
    path = os.path.join(output_base, g)
    print(f"{g}: {len(os.listdir(path))} images")

Processing 942 patients...


/Users/sola/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Feature Extraction Complete ---
Grade_0: 23051 images
Grade_1: 3202 images
Grade_2: 1171 images
Grade_3: 2007 images
